In [4]:
import pandas as pd


# =========================================================
# 1. Load raw data
# =========================================================

print("=" * 60)
print("LOADING RAW DATA")
print("=" * 60)

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


# =========================================================
# 2. Check train/test structure
# =========================================================

print("\n" + "=" * 60)
print("COLUMN CHECK")
print("=" * 60)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

target = "PM2_5_next_hour"

# Target must exist in train
if target not in train.columns:
    raise ValueError(
        "ERROR: PM2_5_next_hour is missing from training data."
    )

# Target should not exist in test
if target in test.columns:
    raise ValueError(
        "ERROR: Test data should not contain PM2_5_next_hour."
    )

# Check that train/test predictors match
train_features = set(train.columns) - {target}
test_features = set(test.columns)

if train_features != test_features:

    print("\nColumns only in train:")
    print(train_features - test_features)

    print("\nColumns only in test:")
    print(test_features - train_features)

    raise ValueError(
        "ERROR: Train and test feature columns do not match."
    )

print("\nTrain/test feature columns match correctly ✅")


# =========================================================
# 3. Preview data
# =========================================================

print("\n" + "=" * 60)
print("FIRST 5 TRAINING ROWS")
print("=" * 60)

display(train.head())


# =========================================================
# 4. Basic information
# =========================================================

print("\n" + "=" * 60)
print("TRAIN INFO")
print("=" * 60)

train.info()

print("\n" + "=" * 60)
print("TEST INFO")
print("=" * 60)

test.info()


# =========================================================
# 5. Missing values BEFORE cleaning
# =========================================================

print("\n" + "=" * 60)
print("MISSING VALUES BEFORE CLEANING")
print("=" * 60)

print("\nMissing values in train:")
print(
    train.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing percentage in train:")
print(
    (train.isna().mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)

print("\nMissing values in test:")
print(
    test.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing percentage in test:")
print(
    (test.isna().mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)


# =========================================================
# 6. Convert timestamp to datetime
# =========================================================

print("\n" + "=" * 60)
print("TIMESTAMP CONVERSION")
print("=" * 60)

train["observation_timestamp"] = pd.to_datetime(
    train["observation_timestamp"],
    errors="raise"
)

test["observation_timestamp"] = pd.to_datetime(
    test["observation_timestamp"],
    errors="raise"
)

print(
    "Train timestamp dtype:",
    train["observation_timestamp"].dtype
)

print(
    "Test timestamp dtype:",
    test["observation_timestamp"].dtype
)

print("Timestamp conversion completed ✅")


# =========================================================
# 7. Duplicate checks
# =========================================================

print("\n" + "=" * 60)
print("DUPLICATE CHECKS")
print("=" * 60)

print("\nFull duplicate rows:")
print("Train:", train.duplicated().sum())
print("Test:", test.duplicated().sum())


train_station_time_duplicates = train.duplicated(
    subset=["station", "observation_timestamp"]
).sum()

test_station_time_duplicates = test.duplicated(
    subset=["station", "observation_timestamp"]
).sum()

print("\nDuplicate station-timestamp pairs:")
print("Train:", train_station_time_duplicates)
print("Test:", test_station_time_duplicates)


# =========================================================
# 8. Numerical summary
# =========================================================

numeric_cols = [
    "current_PM2_5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "TEMP",
    "PRES",
    "DEWP",
    "RAIN",
    "WSPM"
]

print("\n" + "=" * 60)
print("NUMERIC SUMMARY")
print("=" * 60)

display(
    train[numeric_cols]
    .describe()
    .T
)


# =========================================================
# 9. Check impossible negative values
# =========================================================

# TEMP and DEWP are not included here because
# negative temperature and dew point values are possible.

non_negative_cols = [
    "current_PM2_5",
    "PM10",
    "SO2",
    "NO2",
    "CO",
    "O3",
    "RAIN",
    "WSPM"
]

print("\n" + "=" * 60)
print("NEGATIVE VALUE CHECK")
print("=" * 60)

for col in non_negative_cols:

    train_negative = (train[col] < 0).sum()
    test_negative = (test[col] < 0).sum()

    print(
        f"{col}: "
        f"train negatives = {train_negative}, "
        f"test negatives = {test_negative}"
    )


# =========================================================
# 10. Station checks
# =========================================================

print("\n" + "=" * 60)
print("STATION CHECKS")
print("=" * 60)

print("\nNumber of train stations:")
print(train["station"].nunique())

print("\nNumber of test stations:")
print(test["station"].nunique())

print("\nTrain station counts:")
print(train["station"].value_counts())

train_stations = set(train["station"].unique())
test_stations = set(test["station"].unique())

print("\nStations only in train:")
print(train_stations - test_stations)

print("\nStations only in test:")
print(test_stations - train_stations)


# =========================================================
# 11. Wind direction checks
# =========================================================

print("\n" + "=" * 60)
print("WIND DIRECTION CHECK")
print("=" * 60)

print("\nTrain wind directions:")
print(
    train["wd"]
    .value_counts(dropna=False)
)

print("\nTest wind directions:")
print(
    test["wd"]
    .value_counts(dropna=False)
)


# =========================================================
# 12. Timestamp consistency
# =========================================================

print("\n" + "=" * 60)
print("TIMESTAMP CONSISTENCY")
print("=" * 60)

train_time_mismatch = (
    (train["observation_timestamp"].dt.year != train["year"])
    |
    (train["observation_timestamp"].dt.month != train["month"])
    |
    (train["observation_timestamp"].dt.day != train["day"])
    |
    (train["observation_timestamp"].dt.hour != train["hour"])
)

test_time_mismatch = (
    (test["observation_timestamp"].dt.year != test["year"])
    |
    (test["observation_timestamp"].dt.month != test["month"])
    |
    (test["observation_timestamp"].dt.day != test["day"])
    |
    (test["observation_timestamp"].dt.hour != test["hour"])
)

print(
    "Train timestamp mismatches:",
    train_time_mismatch.sum()
)

print(
    "Test timestamp mismatches:",
    test_time_mismatch.sum()
)


# =========================================================
# 13. Train / test time ranges
# =========================================================

print("\n" + "=" * 60)
print("TIME RANGE CHECK")
print("=" * 60)

print("\nTrain time range:")
print(
    train["observation_timestamp"].min(),
    "to",
    train["observation_timestamp"].max()
)

print("\nTest time range:")
print(
    test["observation_timestamp"].min(),
    "to",
    test["observation_timestamp"].max()
)


# =========================================================
# 14. Target checks
# =========================================================

print("\n" + "=" * 60)
print("TARGET CHECK")
print("=" * 60)

print("\nTarget summary:")
display(
    train[target]
    .describe()
)

print(
    "Missing target values:",
    train[target].isna().sum()
)

print(
    "Negative target values:",
    (train[target] < 0).sum()
)


# =========================================================
# 15. Handle missing values
# =========================================================

print("\n" + "=" * 60)
print("MISSING VALUE TREATMENT")
print("=" * 60)


# ---------------------------------------------------------
# 15A. Categorical missing values
# ---------------------------------------------------------

# Missing wind direction becomes its own category.

train["wd"] = train["wd"].fillna("Missing")
test["wd"] = test["wd"].fillna("Missing")

print("\nWind direction NaN -> 'Missing' ✅")


# ---------------------------------------------------------
# 15B. Numerical missing values
# ---------------------------------------------------------

# Calculate column means using TRAINING DATA ONLY.
# These same means will also be used for the test set.

train_means = train[numeric_cols].mean()

print("\nTraining means used for numerical imputation:")
display(train_means)


# Fill missing numerical values in train
train[numeric_cols] = (
    train[numeric_cols]
    .fillna(train_means)
)


# Fill missing numerical values in test
# using the SAME training-set means
test[numeric_cols] = (
    test[numeric_cols]
    .fillna(train_means)
)

print("Numerical NaN -> training-set mean ✅")


# =========================================================
# 16. Missing values AFTER cleaning
# =========================================================

print("\n" + "=" * 60)
print("MISSING VALUES AFTER CLEANING")
print("=" * 60)

print("\nRemaining missing values in train:")
print(
    train.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nRemaining missing values in test:")
print(
    test.isna()
    .sum()
    .sort_values(ascending=False)
)


# =========================================================
# 17. Final validation
# =========================================================

print("\n" + "=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print("Final train shape:", train.shape)
print("Final test shape:", test.shape)


# Target must exist
assert target in train.columns

# Target must contain no missing values
assert train[target].isna().sum() == 0

# Train/test predictor columns must match
assert (
    set(train.columns) - {target}
    ==
    set(test.columns)
)

# No missing values should remain
assert train.isna().sum().sum() == 0
assert test.isna().sum().sum() == 0


print("\nNo missing values remain ✅")
print("Train/test feature columns match ✅")
print("Target is valid ✅")


# =========================================================
# 18. Save cleaned datasets
# =========================================================

print("\n" + "=" * 60)
print("SAVING CLEANED DATA")
print("=" * 60)

train.to_csv(
    "data/train_cleaned.csv",
    index=False
)

test.to_csv(
    "data/test_cleaned.csv",
    index=False
)

print("\nSuccessfully saved:")
print("data/train_cleaned.csv")
print("data/test_cleaned.csv")

print("\nDATA CLEANING COMPLETE ✅")

LOADING RAW DATA
Train shape: (360954, 20)
Test shape: (51063, 19)

COLUMN CHECK

Train columns:
['id', 'observation_timestamp', 'station', 'year', 'month', 'day', 'hour', 'current_PM2_5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'PM2_5_next_hour']

Test columns:
['id', 'observation_timestamp', 'station', 'year', 'month', 'day', 'hour', 'current_PM2_5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM']

Train/test feature columns match correctly ✅

FIRST 5 TRAINING ROWS


,id,observation_timestamp,station,year,month,day,hour,current_PM2_5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,PM2_5_next_hour
0,AQ_249AF8D9D878,2013-03-01 00:00:00,Aotizhongxin,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,8.0
1,AQ_EB620EFFEEC8,2013-03-01 00:00:00,Changping,2013,3,1,0,3.0,6.0,13.0,7.0,300.0,85.0,-2.3,1020.8,-19.7,0.0,E,0.5,3.0
2,AQ_27EEFC23DFA3,2013-03-01 00:00:00,Dingling,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,7.0
3,AQ_C2DEB5BEFD7E,2013-03-01 00:00:00,Dongsi,2013,3,1,0,9.0,9.0,3.0,17.0,300.0,89.0,-0.5,1024.5,-21.4,0.0,NNW,5.7,4.0
4,AQ_8B32BEDFDF0D,2013-03-01 00:00:00,Guanyuan,2013,3,1,0,4.0,4.0,14.0,20.0,300.0,69.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,4.0



TRAIN INFO
<class 'pandas.DataFrame'>
RangeIndex: 360954 entries, 0 to 360953
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     360954 non-null  str    
 1   observation_timestamp  360954 non-null  str    
 2   station                360954 non-null  str    
 3   year                   360954 non-null  int64  
 4   month                  360954 non-null  int64  
 5   day                    360954 non-null  int64  
 6   hour                   360954 non-null  int64  
 7   current_PM2_5          358488 non-null  float64
 8   PM10                   359032 non-null  float64
 9   SO2                    356291 non-null  float64
 10  NO2                    353433 non-null  float64
 11  CO                     345122 non-null  float64
 12  O3                     352455 non-null  float64
 13  TEMP                   360777 non-null  float64
 14  PRES                   360776 non-n

,count,mean,std,min,25%,50%,75%,max
current_PM2_5,358488.0,77.990563,77.618685,2.0000,21.0,55.0000,108.0,999.0
PM10,359032.0,103.524493,88.915230,2.0000,37.0,82.0000,144.0,999.0
SO2,356291.0,16.375999,22.395485,0.2856,3.0,7.7112,20.0,500.0
NO2,353433.0,49.487934,34.374613,1.0265,23.0,42.0000,69.0,290.0
CO,345122.0,1184.508325,1085.439608,100.0000,500.0,900.0000,1500.0,10000.0
O3,352455.0,60.843154,58.110169,0.2142,13.0,49.0000,86.0,1071.0
TEMP,360777.0,14.472746,11.376364,-19.9000,4.4,16.1000,23.9,41.6
PRES,360776.0,1009.756625,10.346585,982.4000,1001.4,1009.0000,1017.8,1042.8
DEWP,360772.0,3.282668,13.830309,-43.4000,-8.1,4.5000,15.8,29.1
RAIN,360780.0,0.066771,0.832099,0.0000,0.0,0.0000,0.0,72.5



NEGATIVE VALUE CHECK
current_PM2_5: train negatives = 0, test negatives = 0
PM10: train negatives = 0, test negatives = 0
SO2: train negatives = 0, test negatives = 0
NO2: train negatives = 0, test negatives = 0
CO: train negatives = 0, test negatives = 0
O3: train negatives = 0, test negatives = 0
RAIN: train negatives = 0, test negatives = 0
WSPM: train negatives = 0, test negatives = 0

STATION CHECKS

Number of train stations:
12

Number of test stations:
12

Train station counts:
station
Wanliu           30393
Guanyuan         30199
Nongzhanguan     30175
Gucheng          30166
Dingling         30141
Tiantan          30102
Wanshouxigong    30091
Dongsi           30079
Changping        29992
Shunyi           29940
Aotizhongxin     29846
Huairou          29830
Name: count, dtype: int64

Stations only in train:
set()

Stations only in test:
set()

WIND DIRECTION CHECK

Train wind directions:
wd
NE     33947
ENE    29634
N      27237
E      26286
NW     25880
SW     24582
NNE    2384

count    360954.000000
mean         78.043979
std          77.777930
min           2.000000
25%          21.000000
50%          55.000000
75%         108.000000
max         999.000000
Name: PM2_5_next_hour, dtype: float64

Missing target values: 0
Negative target values: 0

MISSING VALUE TREATMENT

Wind direction NaN -> 'Missing' ✅

Training means used for numerical imputation:


current_PM2_5      77.990563
PM10              103.524493
SO2                16.375999
NO2                49.487934
CO               1184.508325
O3                 60.843154
TEMP               14.472746
PRES             1009.756625
DEWP                3.282668
RAIN                0.066771
WSPM                1.734617
dtype: float64

Numerical NaN -> training-set mean ✅

MISSING VALUES AFTER CLEANING

Remaining missing values in train:
id                       0
observation_timestamp    0
WSPM                     0
wd                       0
RAIN                     0
DEWP                     0
PRES                     0
TEMP                     0
O3                       0
CO                       0
NO2                      0
SO2                      0
PM10                     0
current_PM2_5            0
hour                     0
day                      0
month                    0
year                     0
station                  0
PM2_5_next_hour          0
dtype: int64

Remaining missing values in test:
id                       0
NO2                      0
wd                       0
RAIN                     0
DEWP                     0
PRES                     0
TEMP                     0
O3                       0
CO                       0
SO2                      0
observation_timestamp    0
PM10       